# AgencyOS — isolated private persistence canary

This notebook is intentionally separate from the model benchmark. Save a private version of this notebook to test persistence without downloading model weights or rerunning Phase 2. No model, customer data, secret, paid service or API is used.


In [ ]:
from pathlib import Path
import hashlib, json, datetime

canary_dir = Path('/kaggle/working/agencyos-checkpoint-canary')
canary_dir.mkdir(parents=True, exist_ok=True)
payload = 'AGENCYOS_PRIVATE_CHECKPOINT_CANARY_V1'
(canary_dir / 'canary.txt').write_text(payload, encoding='utf-8')
manifest = {
    'purpose': 'persistence-only-canary',
    'sha256': hashlib.sha256(payload.encode()).hexdigest(),
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
(canary_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('CANARY CREATED — now Save Version / Save & Run All, keeping this notebook private.')
print(json.dumps(manifest, indent=2))


## Fresh-session verification

After saving the private version, start a new session and attach that version's output as an input. Set `AGENCYOS_VERIFY_PERSISTENCE=1` in the new session before running this cell. It searches attached inputs and fails closed if the file or checksum is missing.


In [ ]:
import os
if os.environ.get('AGENCYOS_VERIFY_PERSISTENCE') != '1':
    print('SKIPPED — set AGENCYOS_VERIFY_PERSISTENCE=1 only in a fresh session with the saved output attached.')
else:
    from pathlib import Path
    import hashlib, json
    matches = list(Path('/kaggle/input').rglob('canary.txt'))
    verified = False
    for item in matches:
        manifest_file = item.parent / 'manifest.json'
        if not manifest_file.exists():
            continue
        content = item.read_text(encoding='utf-8')
        manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
        digest = hashlib.sha256(content.encode()).hexdigest()
        if content == 'AGENCYOS_PRIVATE_CHECKPOINT_CANARY_V1' and digest == manifest.get('sha256'):
            print('PERSISTENCE READ-BACK: PASS')
            print('Attached artifact:', item)
            print('SHA-256:', digest)
            verified = True
            break
    if not verified:
        raise RuntimeError('PERSISTENCE READ-BACK: FAIL — saved private output missing or checksum mismatch.')
